### Cargar librerias

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import seaborn as sns 
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('Datos/Transformados/df_unido.csv')

In [3]:
print(df.shape)
print(df.columns)


(52004, 82)
Index(['booked_at', 'checkin_time', 'checkout_time', 'lead_time',
       'lenght_of_stay', 'checkin_month', 'checkin_day', 'adult_count',
       'child_count', 'origin', 'travel_agency_name', 'requested_category',
       'requested_category_name', 'asset', 'asset_type', 'brand',
       'asset_opening_date', 'available_units', 'business_segment', 'rate',
       'rate_group_name', 'rate_type', 'completed_entry_forms_count',
       'all_entry_forms_completed', 'last_entry_form_completed_at',
       'returning_inhabitant', 'recurrence', 'libere_community',
       'bought_products', 'product_count', 'reservation_net_value',
       'total_adr', 'status', 'cancelled_at', 'cancellation_reason',
       'cancellation_lead_time', 'ciudad', 'mes_cancelacion',
       'estacion_cancelacion', 'mes_estancia', 'estacion_estancia',
       'mes_reserva', 'estacion_reserva', 'dia_semana_reserva',
       'dia_semana_reserva_nombre', 'dia_semana_cancelacion',
       'dia_semana_cancelacion_nombr

In [4]:
#aplicar funcion mas adelante
df['booked_at'] = pd.to_datetime(df['booked_at'], format="mixed", errors="coerce")
df['checkin_time'] = pd.to_datetime(df['checkin_time'], format="mixed", errors="coerce")
df['checkout_time'] = pd.to_datetime(df['checkout_time'], format="mixed", errors="coerce")
df['asset_opening_date'] = pd.to_datetime(df['asset_opening_date'], format="mixed", errors="coerce")
df['cancelled_at'] = pd.to_datetime(df['cancelled_at'], format="mixed", errors="coerce")
df['last_entry_form_completed_at'] = pd.to_datetime(df['last_entry_form_completed_at'], format="mixed", errors="coerce")
df['completed_entry_forms_count'] = pd.to_numeric(df['completed_entry_forms_count'], errors="coerce")

In [5]:
# nuevas columnas con los mes que es
df['checkin_month'] = df['checkin_time'].dt.month
df['checkout_month'] = df['checkout_time'].dt.month

### Filtramos las variables que consideramos relevantes para el clustering.


In [6]:
# 1) Selección de variables (con canal + geo + estacionalidad + meteo)

# 1. Numéricas (Comportamiento y Valor)
cols_num_cluster = [
    'lead_time',                    # Planificador vs Impulsivo
    'lenght_of_stay',               # Escapada vs Vacaciones
    'adult_count', 'child_count',   # Pareja vs Familia vs Solo
    'available_units',              # Ocupación hotel
    'completed_entry_forms_count',  # Digitalización del cliente
    'product_count',                # Gasto extra
    'reservation_net_value',        # Valor total
    'total_adr',                    # Nivel adquisitivo (Precio noche)
    'espera_dias',                  # Paciencia
    'recurrence',                   # Fidelidad
    'ratio_asistencia',             # Fiabilidad histórica
    # Clima (Contexto)
    'tmed', 'prec', 'sol'           # ¿Viaja con buen o mal tiempo?
]

# 2. Binarias (Perfil y Momento)
cols_bin_cluster = [
    'bought_products',      # ¿Gasta en extras?
    'returning_inhabitant', # ¿Repetidor?
    'libere_community',     # ¿Club fidelidad?
    'es_finde',             # CLAVE: Ocio (True) vs Negocio (False)
    'es_festivo'            # CLAVE: Turismo de puente/fiesta
]

# 3. Categóricas (Contexto Fijo)
cols_cat_cluster = [
    'asset_type',       # ¿Hotel, Hostel, Apartamento?
    'business_segment', # Segmento negocio
    'rate_type',        # ¿Tarifa flexible o no? (Riesgo)
    'origin',           # ¿Viene de Booking o Directo?
    'provincia',        # Destino
    'mes_estancia'      # Estacionalidad
]

cols_cluster = cols_num_cluster + cols_bin_cluster + cols_cat_cluster
df_cluster = df[cols_cluster].copy()

### Limpieza de nulos


In [7]:
nulos = df_cluster.isna().sum()
print(nulos[nulos > 0])# no hay nulos para las variables que elijo

Series([], dtype: int64)


In [8]:
# (Opcional) Log1p solo en variables muy sesgadas
for c in ['reservation_net_value', 'total_adr', 'lead_time']:
    if c in df_cluster.columns:
        df_cluster[c] = pd.to_numeric(df_cluster[c], errors='coerce')
        df_cluster[c] = np.log1p(df_cluster[c])

# Binarias -> 0/1 (solo si vienen como texto)
mapeo = {'yes': 1, 'no': 0, 'true': 1, 'false': 0}
for col in cols_bin_cluster:
    if col in df_cluster.columns and df_cluster[col].dtype == 'object':
        df_cluster[col] = (
            df_cluster[col].astype(str).str.lower().map(mapeo).astype(int)
        )


### Dummies

In [9]:
df_cluster = pd.get_dummies(
    df_cluster,
    columns=cols_cat_cluster,
    drop_first=False
)
df_cluster.to_csv('Datos/Transformados/df_model_dummies_cluster.csv', index=False)